# Evaluation results
Load saved episodes and summarise raw returns. No evaluations run here.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


Select completed runs below, using paths relative to this notebook. Include each checkpoint/claim only once to avoid counting reruns twice.


In [ ]:
result_files = [
    # Path("../evaluation_results/run_name/episodes.csv"),
]

if not result_files:
    raise ValueError("Add evaluation CSV paths to result_files first.")

results = pd.concat([pd.read_csv(path) for path in result_files], ignore_index=True)
results.head()


Average episodes within each checkpoint and target first.


In [ ]:
groups = ["claim", "environment", "dataset", "K", "target_return"]

per_model = results.groupby(groups + ["checkpoint", "training_seed"]).agg(
    episodes=("episode_return", "count"),
    mean_return=("episode_return", "mean"),
    episode_std=("episode_return", "std"),
    mean_length=("episode_length", "mean"),
).reset_index()

per_model


Then average the model means across training seeds. `seed_std` is undefined (`NaN`) with one model; it is not episode variability.


In [ ]:
summary = per_model.groupby(groups).agg(
    models=("mean_return", "count"),
    mean_return=("mean_return", "mean"),
    seed_std=("mean_return", "std"),
).reset_index()

summary


## Claim 1  Return conditioning
Compare requested targets with achieved returns, separately for each environment and dataset. Values are raw returns; D4RL normalization is not added yet. Error bars show standard deviation across training-seed means, not confidence intervals. With one model there is no error bar.


In [ ]:
claim1 = summary[summary["claim"] == 1].copy()
claim1["absolute_target_error"] = (claim1["mean_return"] - claim1["target_return"]).abs()
claim1


In [ ]:
for (environment, dataset, k), data in claim1.groupby(["environment", "dataset", "K"]):
    data = data.sort_values("target_return")
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(data["target_return"], data["mean_return"], "o-", label="Achieved return")
    with_std = data.dropna(subset=["seed_std"])
    if not with_std.empty:
        ax.errorbar(with_std["target_return"], with_std["mean_return"],
                    yerr=with_std["seed_std"], fmt="none", capsize=3)
    limits = [min(data["target_return"].min(), data["mean_return"].min()),
              max(data["target_return"].max(), data["mean_return"].max())]
    ax.plot(limits, limits, "--", color="gray", label="Achieved = requested")
    ax.set(title=f"{environment} / {dataset} / K={k}",
           xlabel="Requested return", ylabel="Mean achieved return")
    ax.legend()
    plt.tight_layout()
    plt.show()


A small error means the average achieved return is close to the target. The table below averages this error equally across tested targets; it is not the average absolute error of individual episodes.


In [ ]:
claim1.groupby(["environment", "dataset", "K"]).agg(
    targets=("target_return", "count"),
    mean_absolute_target_error=("absolute_target_error", "mean"),
)


## Claim 2  Context length
Compare K=1 and K=20 at the same target. Keep environments, datasets and targets separate. Error bars again show variability across training-seed means.


In [ ]:
claim2 = summary[summary["claim"] == 2].copy()
claim2


In [ ]:
for (environment, dataset, target), data in claim2.groupby(["environment", "dataset", "target_return"]):
    data = data.sort_values("K")
    positions = list(range(len(data)))
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.bar(positions, data["mean_return"])
    for position, row in zip(positions, data.itertuples()):
        if pd.notna(row.seed_std):
            ax.errorbar(position, row.mean_return, yerr=row.seed_std,
                        fmt="none", color="black", capsize=4)
    ax.set_xticks(positions)
    ax.set_xticklabels([f"K={k}" for k in data["K"]])
    ax.set(title=f"{environment} / {dataset} / target={target:g}",
           ylabel="Mean achieved return")
    plt.tight_layout()
    plt.show()


Pair the two K values by training seed before computing their difference. Positive values favour K=20. Missing pairs remain visible as NaN and do not contribute to the average. Use the same episode seeds for both models.


In [ ]:
paired = per_model[per_model["claim"] == 2].pivot(
    index=["environment", "dataset", "target_return", "training_seed"],
    columns="K", values="mean_return",
).reindex(columns=[1, 20])
paired["difference_K20_minus_K1"] = paired[20] - paired[1]
paired


In [ ]:
context_comparison = paired.groupby(level=["environment", "dataset", "target_return"]).agg(
    paired_seeds=("difference_K20_minus_K1", "count"),
    mean_difference=("difference_K20_minus_K1", "mean"),
    seed_std=("difference_K20_minus_K1", "std"),
)
context_comparison
